# Étude de cas 3 — Jalon 4 : le rythme est une séquence

**Objectif :** comparer des modèles qui lisent un ECG comme une suite ordonnée dans le temps. Le signal brut contient 1 250 mesures consécutives pour 10 secondes. Nous comparerons un RNN simple, un LSTM, un hybride **CNN + LSTM/GRU** (convolutions temporelles suivies d’une couche récurrente) et un GRU alimenté par des battements détectés.

## Parcours de travail

1. Charger Holter A, comprendre la forme temporelle et préparer les entrées sans ouvrir le test.
2. Entraîner un RNN simple puis un LSTM sur le signal brut ; suivre les gradients.
3. Comparer un CNN seul à un hybride CNN-LSTM/GRU.
4. Détecter les QRS, former une séquence de battements, puis entraîner un GRU avec masquage.
5. Comparer F1 macro par classe, paramètres et temps d’entraînement ; vérifier les calculs théoriques.
6. Choisir sur validation, puis mesurer le test une seule fois.

Les données ECG sont simulées : elles ne permettent aucune conclusion médicale.

## 1. Vérifier l’environnement

In [3]:
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

for paquet in ['numpy', 'scipy', 'scikit-learn', 'tensorflow', 'matplotlib']:
    try:
        print(f'{paquet}: {version(paquet)}')
    except PackageNotFoundError:
        print(f'{paquet}: NON INSTALLÉ')

numpy: 2.5.3
scipy: 1.18.1
scikit-learn: 1.9.1
tensorflow: 2.21.0
matplotlib: 3.11.2


## 2. Charger Holter A sans toucher au test

Chaque fenêtre contient 1 250 échantillons, soit 10 secondes à 125 Hz. Les ensembles fournis sont séparés par patient ; on conserve cette séparation pour que des fenêtres de la même personne ne soient pas à la fois dans l’apprentissage et la validation.

In [5]:
from pathlib import Path
import numpy as np

# Trouver la racine PRACTICE même si le notebook est lancé depuis notebooks/Jalon4.
candidats = [Path.cwd(), *Path.cwd().parents]
PRACTICE_DIR = next((p for p in candidats if (p / 'data' / 'holter_A').is_dir()), None)
if PRACTICE_DIR is None:
    raise FileNotFoundError('Lancer ce notebook depuis PRACTICE ou un sous-dossier.')
DELIVERABLE_DIR = PRACTICE_DIR / 'deliverables' / 'Jalon4'
DELIVERABLE_DIR.mkdir(parents=True, exist_ok=True)

# Charger seulement train et validation ; le test sera réservé à la toute dernière étape.
cles = ['X_train', 'classe_train', 'patient_train', 'X_val', 'classe_val', 'patient_val']
with np.load(PRACTICE_DIR / 'data' / 'holter_A' / 'uc3_holterA.npz') as f:
    A = {cle: f[cle].copy() for cle in cles}

for nom in ['train', 'val']:
    X, y, patients = A[f'X_{nom}'], A[f'classe_{nom}'], A[f'patient_{nom}']
    print(f'{nom}: {len(y)} fenêtres ; signal {X.shape} ; {len(np.unique(patients))} patients')
    print('  classes :', dict(zip(*np.unique(y, return_counts=True))))
assert not (set(np.unique(A['patient_train'])) & set(np.unique(A['patient_val'])))
print('Vérifié : aucun patient partagé entre train et validation.')

train: 8400 fenêtres ; signal (8400, 1250) ; 280 patients
  classes : {np.int8(0): np.int64(4348), np.int8(1): np.int64(1209), np.int8(2): np.int64(1018), np.int8(3): np.int64(995), np.int8(4): np.int64(830)}
val: 1800 fenêtres ; signal (1800, 1250) ; 60 patients
  classes : {np.int8(0): np.int64(892), np.int8(1): np.int64(238), np.int8(2): np.int64(292), np.int8(3): np.int64(198), np.int8(4): np.int64(180)}
Vérifié : aucun patient partagé entre train et validation.


## 3. Comprendre la forme temporelle et normaliser les fenêtres

Une fenêtre n’est pas une image ni une liste de caractéristiques : c’est une mesure ECG qui avance dans le temps. L’axe des 1 250 positions correspond aux instants successifs ; le dernier axe de taille 1 indique un seul canal ECG. Le RNN recevra donc chaque tension une par une, dans l’ordre.

In [7]:
FS = 125
LONGUEUR_FENETRE = 1250
DUREE_SECONDES = LONGUEUR_FENETRE / FS

def normaliser_fenetres(X):
    # Mettre chaque fenêtre à moyenne 0 et écart-type 1 séparément.
    X = X.astype('float32')
    moyenne = X.mean(axis=1, keepdims=True)
    ecart_type = X.std(axis=1, keepdims=True)
    return ((X - moyenne) / (ecart_type + 1e-6))[..., None]

SA = {s: normaliser_fenetres(A[f'X_{s}']) for s in ['train', 'val']}
print('Durée d’une fenêtre :', DUREE_SECONDES, 'secondes')
print('Entrée train pour un RNN :', SA['train'].shape, '= (fenêtres, temps, canaux)')
print('Entrée validation :', SA['val'].shape)
print('Exemple : classe', int(A['classe_train'][0]), ', patient', int(A['patient_train'][0]))
print('10 premières mesures après normalisation :', np.round(SA['train'][0, :10, 0], 3))

Durée d’une fenêtre : 10.0 secondes
Entrée train pour un RNN : (8400, 1250, 1) = (fenêtres, temps, canaux)
Entrée validation : (1800, 1250, 1)
Exemple : classe 0 , patient 0
10 premières mesures après normalisation : [-0.395 -0.158 -0.486 -0.23  -0.527 -0.634 -0.669 -0.569 -1.038 -0.252]


## 4. Idée d’un RNN simple

Un RNN lit les mesures dans l’ordre : à chaque instant, il reçoit la tension actuelle et un **état caché** résumé des instants précédents. Il met à jour cet état puis passe au point suivant. À la fin des 1 250 pas, une couche dense utilise le dernier état pour choisir l’une des cinq classes.

Version simplifiée de la mise à jour : **état actuel = tanh(signal actuel × poids + état précédent × poids récurrent + biais)**. Le modèle doit donc transmettre l’information à travers une longue chaîne temporelle ; le gradient peut devenir très petit en remontant cette chaîne.

## 5. Construire le RNN brut de référence

Pour commencer, nous prenons seulement 16 unités récurrentes. Une unité est une petite valeur de l’état caché. Le modèle reçoit `(1250, 1)` et renvoie cinq scores de classe. Cette cellule construit et compile le réseau ; elle ne l’entraîne pas encore.

In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, recall_score
from time import perf_counter

GRAINE = 42
keras.utils.set_random_seed(GRAINE)
CLASSES = np.arange(5)

rnn_simple = keras.Sequential([
    keras.Input(shape=(LONGUEUR_FENETRE, 1), name='ecg_temps'),
    # L’unité récurrente met à jour son état caché à chacun des 1 250 instants.
    layers.SimpleRNN(16, activation='tanh', name='rnn_simple'),
    # La sortie transforme l’état final en cinq probabilités de rythme.
    layers.Dense(5, activation='softmax', name='rythme'),
], name='rnn_brut')

rnn_simple.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
print('Forme attendue :', rnn_simple.input_shape)
print('Nombre total de paramètres :', rnn_simple.count_params())
rnn_simple.summary()

/home/mapster/.config/matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /tmp/matplotlib-e1o4ndqv because there was an issue with the default path (/home/mapster/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Forme attendue : (None, 1250, 1)
Nombre total de paramètres : 373
Model: "rnn_brut"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rnn_simple (SimpleRNN)          │ (None, 16)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rythme (Dense)                  │ (None, 5)              │            85 │
└─────────────────────────────────┴─────────────

## 6. Mesurer les gradients pendant l’apprentissage

La rétropropagation calcule comment chaque poids devrait changer pour réduire l’erreur. Une norme très petite signifie que le poids reçoit peu de signal d’apprentissage. En plus des normes par variable (comme dans le code de départ), nous mesurerons le gradient de l’erreur par rapport à l’entrée, regroupé en tranches temporelles. Cela permet de voir si les points éloignés dans le passé influencent encore la décision.

In [12]:
# Un petit lot fixe sert uniquement au diagnostic de gradient, jamais à la validation.
X_DIAG = tf.convert_to_tensor(SA['train'][:32])
y_DIAG = tf.convert_to_tensor(A['classe_train'][:32].astype('int32'))

class DiagnosticGradientRNN(keras.callbacks.Callback):
    def __init__(self, X, y, journal_csv):
        super().__init__()
        self.X, self.y, self.journal_csv = X, y, journal_csv
        self.secondes = []
        self.historique = []
        self.debut_epoque = None
        self.mesure_pren_val = False

    def on_epoch_begin(self, epoch, logs=None):
        self.debut_epoque = perf_counter()
        self.mesure_pren_val = False

    def on_test_begin(self, logs=None):
        # Keras commence la validation après la phase d’entraînement de l’époque.
        if not self.mesure_pren_val:
            self.secondes.append(perf_counter() - self.debut_epoque)
            self.mesure_pren_val = True

    def on_epoch_end(self, epoch, logs=None):
        if not self.mesure_pren_val:
            self.secondes.append(perf_counter() - self.debut_epoque)
        # Calculer une fois les gradients sur le même mini-lot pour comparer les époques.
        with tf.GradientTape(persistent=True) as tape:
            tape.watch(self.X)
            probabilites = self.model(self.X, training=False)
            perte = tf.reduce_mean(
                keras.losses.sparse_categorical_crossentropy(self.y, probabilites))
        gradients_poids = tape.gradient(perte, self.model.trainable_variables)
        gradient_entree = tape.gradient(perte, self.X)
        del tape

        normes = {
            variable.path: float(tf.norm(gradient).numpy())
            for variable, gradient in zip(self.model.trainable_variables, gradients_poids)
            if gradient is not None
        }
        # Moyenne de l’amplitude du gradient par tranche de 125 pas (une seconde).
        profil = tf.reduce_mean(tf.abs(gradient_entree), axis=(0, 2)).numpy()
        par_seconde = [float(profil[i:i+FS].mean()) for i in range(0, len(profil), FS)]
        self.historique.append({'epoque': epoch + 1, 'normes': normes,
                                'gradient_par_seconde': par_seconde})
        print(f'Epoque {epoch+1} — perte={logs["loss"]:.3f}, '
              f'F1 macro val={logs.get("val_f1_macro", float("nan")):.3f}')
        print('  Normes des poids :', {k: round(v, 6) for k, v in normes.items()})
        print('  Gradient moyen entrée, secondes 0→10 :',
              [round(v, 8) for v in par_seconde])

# Ajouter la métrique F1 macro en validation (calculée sur les prédictions de l’époque).
class F1MacroValidation(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        proba = self.model.predict(SA['val'], batch_size=256, verbose=0)
        logs['val_f1_macro'] = f1_score(A['classe_val'], proba.argmax(axis=1),
                                         average='macro', zero_division=0)

poids_classes = compute_class_weight('balanced', classes=CLASSES,
                                     y=A['classe_train'].astype(int))
POIDS_CLASSES = dict(zip(CLASSES.tolist(), poids_classes.tolist()))
print('Poids de classe :', {k: round(v, 3) for k,v in POIDS_CLASSES.items()})

Poids de classe : {0: 0.386, 1: 1.39, 2: 1.65, 3: 1.688, 4: 2.024}


## 7. Premier entraînement et lecture des résultats

On lance trois époques pour observer le comportement sans attendre un entraînement long. La validation utilise 60 patients séparés du train. Ce premier passage est exploratoire ; nous garderons la même durée pour comparer le RNN et le LSTM ensuite, et nous consignerons cette durée dans le journal.

In [14]:
JOURNAL = DELIVERABLE_DIR / 'journal_experiences_jalon4.csv'
diag_rnn = DiagnosticGradientRNN(X_DIAG, y_DIAG, JOURNAL)
callback_f1 = F1MacroValidation()

historique_rnn = rnn_simple.fit(
    SA['train'], A['classe_train'].astype('int32'),
    validation_data=(SA['val'], A['classe_val'].astype('int32')),
    class_weight=POIDS_CLASSES,
    epochs=3, batch_size=128, verbose=0,
    callbacks=[callback_f1, diag_rnn],
)

proba_val_rnn = rnn_simple.predict(SA['val'], batch_size=256, verbose=0)
pred_val_rnn = proba_val_rnn.argmax(axis=1)
F1_RNN_VAL = f1_score(A['classe_val'], pred_val_rnn, average='macro', zero_division=0)
ACC_RNN_VAL = accuracy_score(A['classe_val'], pred_val_rnn)
RAPPELS_RNN_VAL = recall_score(A['classe_val'], pred_val_rnn, labels=CLASSES,
                              average=None, zero_division=0)
TEMPS_EPOQUE_RNN = float(np.mean(diag_rnn.secondes))
print('RNN simple — validation Holter A :')
print('Exactitude :', round(ACC_RNN_VAL, 3), '; F1 macro :', round(F1_RNN_VAL, 3))
print('Rappels classes 0 à 4 :', np.round(RAPPELS_RNN_VAL, 3))
print('Paramètres :', rnn_simple.count_params(),
      '; secondes par époque :', round(TEMPS_EPOQUE_RNN, 2))

# Ajouter cette expérience au journal CSV pour garder trace des essais.
import csv
with JOURNAL.open(encoding='utf-8-sig', newline='') as f:
    lignes_journal = list(csv.DictReader(f))
champs_journal = ['jalon','experience','graine','f1_macro_validation','exactitude_validation',
                  'rappel_classes_0_a_4','parametres','secondes_par_epoque',
                  'observation_gradient','statut','notes']
lignes_journal = [r for r in lignes_journal if r['experience'] != 'RNN_simple_signal_brut']
lignes_journal.append({
    'jalon':4, 'experience':'RNN_simple_signal_brut', 'graine':GRAINE,
    'f1_macro_validation':round(float(F1_RNN_VAL),4),
    'exactitude_validation':round(float(ACC_RNN_VAL),4),
    'rappel_classes_0_a_4':' / '.join(f'{v:.3f}' for v in RAPPELS_RNN_VAL),
    'parametres':rnn_simple.count_params(),
    'secondes_par_epoque':round(TEMPS_EPOQUE_RNN,2),
    'observation_gradient':'normes par poids et profil temporel tracés à chaque époque',
    'statut':'exploratoire, 3 époques',
    'notes':'Holter A ; entrée signal brut normalisé ; classe équilibrée par poids'})
with JOURNAL.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=champs_journal)
    writer.writeheader(); writer.writerows(lignes_journal)

Epoque 1 — perte=1.639, F1 macro val=0.169
  Normes des poids : {'rnn_brut/rnn_simple/simple_rnn_cell/kernel': 0.155436, 'rnn_brut/rnn_simple/simple_rnn_cell/recurrent_kernel': 0.368299, 'rnn_brut/rnn_simple/simple_rnn_cell/bias': 0.895269, 'rnn_brut/rythme/kernel': 0.185717, 'rnn_brut/rythme/bias': 0.615653}
  Gradient moyen entrée, secondes 0→10 : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.4e-07, 0.00064752]
Epoque 2 — perte=1.615, F1 macro val=0.180
  Normes des poids : {'rnn_brut/rnn_simple/simple_rnn_cell/kernel': 0.166137, 'rnn_brut/rnn_simple/simple_rnn_cell/recurrent_kernel': 0.342075, 'rnn_brut/rnn_simple/simple_rnn_cell/bias': 0.893001, 'rnn_brut/rythme/kernel': 0.199334, 'rnn_brut/rythme/bias': 0.616522}
  Gradient moyen entrée, secondes 0→10 : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5e-07, 0.00051031]
Epoque 3 — perte=1.611, F1 macro val=0.186
  Normes des poids : {'rnn_brut/rnn_simple/simple_rnn_cell/kernel': 0.176866, 'rnn_brut/rnn_simple/simple_rnn_cell/recurrent_kern

## 8. Pourquoi essayer un LSTM ?

Un RNN simple transporte son état d’un instant au suivant. Sur une fenêtre de 1 250 mesures, une information ancienne peut devenir difficile à conserver. Le LSTM ajoute des portes qui apprennent quoi garder, ajouter ou oublier. Il reçoit exactement la même séquence ECG que le RNN simple.

**À comprendre :** le LSTM est toujours un modèle récurrent ; ses portes sont un mécanisme pour mieux gérer la mémoire, pas une nouvelle donnée ni une nouvelle tâche.


## 9. Construire le LSTM sur le signal brut

On remplace seulement `SimpleRNN(16)` par `LSTM(16)`. La couche dense prédit toujours les cinq classes. Comptez les paramètres : le LSTM en a davantage, car il apprend quatre transformations internes (portes) au lieu d’une seule.


In [17]:
# Fixer une graine différente mais reproductible pour cette expérience.
GRAINE_LSTM = 43
keras.utils.set_random_seed(GRAINE_LSTM)

modele_lstm = keras.Sequential([
    keras.Input(shape=(LONGUEUR_FENETRE, 1), name='ecg_temps'),
    # Le LSTM lit les 1 250 mesures et résume la séquence dans son état final.
    layers.LSTM(16, name='lstm'),
    # Produire une probabilité pour chacun des cinq rythmes.
    layers.Dense(5, activation='softmax', name='rythme'),
], name='lstm_signal_brut')

modele_lstm.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
print('Forme attendue :', modele_lstm.input_shape)
print('Nombre total de paramètres :', modele_lstm.count_params())
modele_lstm.summary()


Forme attendue : (None, 1250, 1)
Nombre total de paramètres : 1237
Model: "lstm_signal_brut"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 16)             │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rythme (Dense)                  │ (None, 5)              │            85 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 1,237 (4.83 KB)
 Trainable params: 1,237 (4.83 KB)
 Non-trainable params: 0 (0.00 B)


## 10. Entraîner et comparer au RNN simple

Nous gardons les mêmes données train/validation, les mêmes poids de classes, la même taille de lot et trois époques. Le diagnostic de gradient examine aussi les mêmes 32 fenêtres fixes. La validation sert à comparer les modèles ; le jeu test reste fermé.


In [19]:
diag_lstm = DiagnosticGradientRNN(X_DIAG, y_DIAG, JOURNAL)
callback_f1_lstm = F1MacroValidation()

historique_lstm = modele_lstm.fit(
    SA['train'], A['classe_train'].astype('int32'),
    validation_data=(SA['val'], A['classe_val'].astype('int32')),
    class_weight=POIDS_CLASSES,
    epochs=3, batch_size=128, verbose=0,
    callbacks=[callback_f1_lstm, diag_lstm],
)

proba_val_lstm = modele_lstm.predict(SA['val'], batch_size=256, verbose=0)
pred_val_lstm = proba_val_lstm.argmax(axis=1)
F1_LSTM_VAL = f1_score(A['classe_val'], pred_val_lstm, average='macro', zero_division=0)
ACC_LSTM_VAL = accuracy_score(A['classe_val'], pred_val_lstm)
RAPPELS_LSTM_VAL = recall_score(A['classe_val'], pred_val_lstm, labels=CLASSES,
                               average=None, zero_division=0)
TEMPS_EPOQUE_LSTM = float(np.mean(diag_lstm.secondes))
print('LSTM — validation Holter A :')
print('Exactitude :', round(ACC_LSTM_VAL, 3), '; F1 macro :', round(F1_LSTM_VAL, 3))
print('Rappels classes 0 à 4 :', np.round(RAPPELS_LSTM_VAL, 3))
print('Paramètres :', modele_lstm.count_params(),
      '; secondes par époque :', round(TEMPS_EPOQUE_LSTM, 2))
print('Comparaison F1 macro — RNN :', round(F1_RNN_VAL, 3),
      '| LSTM :', round(F1_LSTM_VAL, 3))
print('Comparaison temps/époque — RNN :', round(TEMPS_EPOQUE_RNN, 2),
      's | LSTM :', round(TEMPS_EPOQUE_LSTM, 2), 's')

# Ajouter les résultats au journal sans effacer l'expérience RNN.
import csv
with JOURNAL.open(encoding='utf-8-sig', newline='') as f:
    lignes_journal = list(csv.DictReader(f))
champs_journal = ['jalon','experience','graine','f1_macro_validation','exactitude_validation',
                  'rappel_classes_0_a_4','parametres','secondes_par_epoque',
                  'observation_gradient','statut','notes']
lignes_journal = [r for r in lignes_journal if r['experience'] != 'LSTM_signal_brut']
lignes_journal.append({
    'jalon':4, 'experience':'LSTM_signal_brut', 'graine':GRAINE_LSTM,
    'f1_macro_validation':round(float(F1_LSTM_VAL),4),
    'exactitude_validation':round(float(ACC_LSTM_VAL),4),
    'rappel_classes_0_a_4':' / '.join(f'{v:.3f}' for v in RAPPELS_LSTM_VAL),
    'parametres':modele_lstm.count_params(),
    'secondes_par_epoque':round(TEMPS_EPOQUE_LSTM,2),
    'observation_gradient':'normes par poids et profil temporel tracés à chaque époque',
    'statut':'exploratoire, 3 époques',
    'notes':'Holter A ; même signal brut normalisé et protocole que le RNN simple'})
with JOURNAL.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=champs_journal)
    writer.writeheader(); writer.writerows(lignes_journal)


Epoque 1 — perte=1.605, F1 macro val=0.179
  Normes des poids : {'lstm_signal_brut/lstm/lstm_cell/kernel': 0.076826, 'lstm_signal_brut/lstm/lstm_cell/recurrent_kernel': 0.060369, 'lstm_signal_brut/lstm/lstm_cell/bias': 0.872819, 'lstm_signal_brut/rythme/kernel': 0.041012, 'lstm_signal_brut/rythme/bias': 0.618597}
  Gradient moyen entrée, secondes 0→10 : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.456e-05]
Epoque 2 — perte=1.600, F1 macro val=0.162
  Normes des poids : {'lstm_signal_brut/lstm/lstm_cell/kernel': 0.190967, 'lstm_signal_brut/lstm/lstm_cell/recurrent_kernel': 0.155543, 'lstm_signal_brut/lstm/lstm_cell/bias': 1.296017, 'lstm_signal_brut/rythme/kernel': 0.077202, 'lstm_signal_brut/rythme/bias': 0.628512}
  Gradient moyen entrée, secondes 0→10 : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.391e-05]
Epoque 3 — perte=1.587, F1 macro val=0.205
  Normes des poids : {'lstm_signal_brut/lstm/lstm_cell/kernel': 0.427556, 'lstm_signal_brut/lstm/lstm_cell/recurrent_kernel': 0.718

## 11. Ce qu’il faut retenir de cette comparaison

Comparez d’abord le **F1 macro** : il résume la qualité sur les cinq classes en leur donnant le même poids. Vérifiez ensuite les rappels par classe, car un score moyen peut cacher une classe mal détectée. Le temps par époque et le nombre de paramètres indiquent le coût de calcul et la taille du modèle.

Ces trois époques sont un **essai exploratoire**, pas encore le résultat final du jalon. Une amélioration du LSTM serait compatible avec l’idée que ses portes aident à conserver l’information, mais il faut aussi comparer les gradients et les modèles suivants avant de conclure.

**Prochaine section :** construire un CNN suivi d’une couche récurrente (LSTM ou GRU), puis le comparer au CNN seul.


## 12. Ce que demande le sujet : CNN seul contre CNN + couche récurrente

Le PDF demande un CNN qui **extrait des motifs locaux et réduit la longueur de la séquence**, puis un LSTM ou GRU qui **modélise l’ordre de ces motifs**. Il faut comparer ce modèle hybride au CNN seul.

Les deux modèles ci-dessous ont le même extracteur convolutionnel et la même entrée. Le CNN seul résume ensuite les caractéristiques ; le modèle hybride les lit encore dans le temps avec un LSTM. Nous choisissons LSTM pour cette comparaison.


## 13. Construire les deux architectures comparables

À chaque fenêtre d’entrée `(1250, 1)`, les convolutions apprennent des motifs voisins du signal. Deux max-poolings divisent la longueur approximativement par quatre : de 1 250 pas à 312. Le CNN seul moyenne les caractéristiques restantes ; le modèle hybride conserve leur ordre avec un LSTM à 16 unités.

Les réseaux sont entraînés séparément : même structure convolutionnelle, mais poids appris indépendamment.


In [23]:
def construire_extracteur_cnn(entree, prefixe):
    # Détecter des motifs ECG locaux, puis réduire graduellement l’axe temporel.
    x = layers.Conv1D(32, 7, padding='same', activation='relu', name=f'{prefixe}_conv1')(entree)
    x = layers.MaxPooling1D(2, name=f'{prefixe}_pool1')(x)
    x = layers.Conv1D(64, 5, padding='same', activation='relu', name=f'{prefixe}_conv2')(x)
    x = layers.MaxPooling1D(2, name=f'{prefixe}_pool2')(x)
    return x

# Modèle de comparaison : CNN, sans couche récurrente.
entree_cnn = keras.Input(shape=(LONGUEUR_FENETRE, 1), name='ecg_temps_cnn')
caracteristiques_cnn = construire_extracteur_cnn(entree_cnn, 'cnn_seul')
x_cnn = layers.GlobalAveragePooling1D(name='resume_temporel')(caracteristiques_cnn)
sortie_cnn = layers.Dense(5, activation='softmax', name='rythme_cnn')(x_cnn)
modele_cnn = keras.Model(entree_cnn, sortie_cnn, name='cnn_seul')

# Modèle demandé : même extracteur local, puis le LSTM lit les caractéristiques dans l’ordre.
entree_hybride = keras.Input(shape=(LONGUEUR_FENETRE, 1), name='ecg_temps_hybride')
caracteristiques_hybride = construire_extracteur_cnn(entree_hybride, 'cnn_hybride')
x_hybride = layers.LSTM(16, name='lstm_succession')(caracteristiques_hybride)
sortie_hybride = layers.Dense(5, activation='softmax', name='rythme_hybride')(x_hybride)
modele_cnn_lstm = keras.Model(entree_hybride, sortie_hybride, name='cnn_suivi_lstm')

for modele in [modele_cnn, modele_cnn_lstm]:
    modele.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                   loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print(f'\n{modele.name} — paramètres : {modele.count_params():,}')
    print('Séquence après convolutions/pooling :', caracteristiques_cnn.shape if modele is modele_cnn else caracteristiques_hybride.shape)



cnn_seul — paramètres : 10,885
Séquence après convolutions/pooling : (None, 312, 64)

cnn_suivi_lstm — paramètres : 15,829
Séquence après convolutions/pooling : (None, 312, 64)


## 14. Entraîner la comparaison exigée

Nous gardons la même validation, les mêmes poids de classes, Adam, taille de lot 128 et trois époques exploratoires. Le score principal est le F1 macro ; les rappels par classe montrent les rythmes éventuellement manqués. Le jeu test n’est pas utilisé.


In [25]:
def entrainer_comparer(modele, nom, graine):
    # Garder des initialisations reproductibles distinctes pour chaque expérience.
    keras.utils.set_random_seed(graine)
    diagnostic = DiagnosticGradientRNN(X_DIAG, y_DIAG, JOURNAL)
    callback_f1_modele = F1MacroValidation()
    modele.fit(
        SA['train'], A['classe_train'].astype('int32'),
        validation_data=(SA['val'], A['classe_val'].astype('int32')),
        class_weight=POIDS_CLASSES, epochs=3, batch_size=128, verbose=0,
        callbacks=[callback_f1_modele, diagnostic],
    )
    probabilites = modele.predict(SA['val'], batch_size=256, verbose=0)
    predictions = probabilites.argmax(axis=1)
    f1 = f1_score(A['classe_val'], predictions, average='macro', zero_division=0)
    exactitude = accuracy_score(A['classe_val'], predictions)
    rappels = recall_score(A['classe_val'], predictions, labels=CLASSES,
                           average=None, zero_division=0)
    temps = float(np.mean(diagnostic.secondes))
    print(f'\n{nom} — validation Holter A')
    print('F1 macro :', round(f1, 3), '; exactitude :', round(exactitude, 3))
    print('Rappels classes 0 à 4 :', np.round(rappels, 3))
    print('Paramètres :', modele.count_params(), '; secondes/époque :', round(temps, 2))
    return {'jalon':4, 'experience':nom, 'graine':graine,
            'f1_macro_validation':round(float(f1),4),
            'exactitude_validation':round(float(exactitude),4),
            'rappel_classes_0_a_4':' / '.join(f'{v:.3f}' for v in rappels),
            'parametres':modele.count_params(),
            'secondes_par_epoque':round(temps,2),
            'observation_gradient':'normes par poids et profil temporel tracés à chaque époque',
            'statut':'exploratoire, 3 époques',
            'notes':'Holter A ; mêmes fenêtres, extracteur convolutif, validation et protocole'}

resultat_cnn = entrainer_comparer(modele_cnn, 'CNN_seul', 44)
resultat_cnn_lstm = entrainer_comparer(modele_cnn_lstm, 'CNN_suivi_LSTM', 45)
print('\nF1 macro — RNN brut :', round(F1_RNN_VAL,3),
      '| LSTM brut :', round(F1_LSTM_VAL,3),
      '| CNN seul :', resultat_cnn['f1_macro_validation'],
      '| CNN + LSTM :', resultat_cnn_lstm['f1_macro_validation'])

# Ajouter les deux expériences au journal en remplaçant leurs éventuelles anciennes lignes.
import csv
with JOURNAL.open(encoding='utf-8-sig', newline='') as f:
    lignes_journal = list(csv.DictReader(f))
champs_journal = ['jalon','experience','graine','f1_macro_validation','exactitude_validation',
                  'rappel_classes_0_a_4','parametres','secondes_par_epoque',
                  'observation_gradient','statut','notes']
noms = {resultat_cnn['experience'], resultat_cnn_lstm['experience']}
lignes_journal = [r for r in lignes_journal if r['experience'] not in noms]
lignes_journal.extend([resultat_cnn, resultat_cnn_lstm])
with JOURNAL.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=champs_journal)
    writer.writeheader(); writer.writerows(lignes_journal)


Epoque 1 — perte=1.589, F1 macro val=0.306
  Normes des poids : {'cnn_seul_conv1/kernel': 0.388864, 'cnn_seul_conv1/bias': 0.282037, 'cnn_seul_conv2/kernel': 0.80917, 'cnn_seul_conv2/bias': 0.526354, 'rythme_cnn/kernel': 0.546655, 'rythme_cnn/bias': 0.605184}
  Gradient moyen entrée, secondes 0→10 : [1.145e-05, 1.143e-05, 1.149e-05, 1.18e-05, 1.198e-05, 1.161e-05, 1.167e-05, 1.169e-05, 1.185e-05, 1.14e-05]
Epoque 2 — perte=1.513, F1 macro val=0.367
  Normes des poids : {'cnn_seul_conv1/kernel': 0.575589, 'cnn_seul_conv1/bias': 0.436554, 'cnn_seul_conv2/kernel': 1.186444, 'cnn_seul_conv2/bias': 0.555185, 'rythme_cnn/kernel': 1.188135, 'rythme_cnn/bias': 0.59364}
  Gradient moyen entrée, secondes 0→10 : [3.039e-05, 3.032e-05, 3.057e-05, 3.115e-05, 3.198e-05, 3.117e-05, 3.096e-05, 3.118e-05, 3.189e-05, 3.055e-05]
Epoque 3 — perte=1.411, F1 macro val=0.413
  Normes des poids : {'cnn_seul_conv1/kernel': 0.889799, 'cnn_seul_conv1/bias': 0.679414, 'cnn_seul_conv2/kernel': 1.540563, 'cnn_seul_

## 15. Interpréter sans conclure trop tôt

Comparez le CNN + LSTM au CNN seul : le seul changement conceptuel est l’ajout du LSTM après l’extracteur convolutif. Si l’hybride progresse, cela suggère que l’ordre temporel des motifs apporte de l’information. Vérifiez toujours les rappels par classe et le coût en paramètres et en temps.

Trois époques donnent une première lecture seulement. La décision finale viendra après l’expérience GRU sur suite de battements et la comparaison des trois familles demandées.


## 16. Troisième approche demandée : GRU sur une suite de battements

Le sujet demande de représenter chaque fenêtre par **au plus 40 intervalles RR et amplitudes QRS**, compléter avec des zéros, puis utiliser un GRU avec `Masking`. Cette entrée n’est plus le signal complet : elle résume chaque battement par deux nombres.

**Limite à signaler :** le notebook de démarrage `uc3_demarrage.ipynb`, qui devait fournir le détecteur QRS officiel, est absent des fichiers disponibles. Pour avancer, nous réutilisons le détecteur pédagogique reconstruit dans le notebook Jalon 1. Il repère des QRS approximatifs ; les résultats de cette expérience dépendent donc de cette reconstruction et doivent être présentés avec cette réserve.


## 17. Détecter les QRS et construire les suites

Pour chaque signal brut Holter A, le détecteur cherche des pics QRS. L’intervalle RR est le temps entre deux pics consécutifs : nombre d’échantillons divisé par 125 Hz. L’amplitude est la valeur du signal en mV au second pic de chaque paire, comme dans le code de démarrage. Nous construisons uniquement train et validation ; le test reste fermé.


In [29]:
# SciPy fournit le filtre numérique et les fonctions de détection de pics.
from scipy.signal import butter, sosfiltfilt, find_peaks

def detect_qrs(window, fs=125):
    # Vérifier que le signal est un tableau numérique à une dimension.
    window = np.asarray(window, dtype=float)

    # Filtre passe-bande : conserver les fréquences utiles pour faire ressortir les QRS.
    sos = butter(3, [5, 18], btype='bandpass', fs=fs, output='sos')
    filtered = sosfiltfilt(sos, window)

    # Après mise au carré, les variations rapides deviennent de fortes valeurs positives.
    # np.diff calcule les variations entre mesures consécutives.
    derivative = np.diff(filtered, prepend=filtered[0])
    # Le carré supprime le signe et fait ressortir les variations rapides.
    energy = derivative ** 2
    # Moyenner l’énergie sur environ 0,12 seconde pour lisser les petites fluctuations.
    smooth_n = max(1, round(0.12 * fs))
    energy_smooth = np.convolve(energy, np.ones(smooth_n) / smooth_n, mode='same')

    # La MAD est moins influencée que l’écart-type par quelques pics d’énergie très élevés.
    med = np.median(energy_smooth)
    mad = np.median(np.abs(energy_smooth - med))
    threshold = med + 4 * mad
    # Détecter les pics d’énergie en les espaçant d’au moins 0,25 seconde.
    candidates, _ = find_peaks(
        energy_smooth,
        distance=round(0.25 * fs),
        height=threshold,
        prominence=max(4 * mad, np.finfo(float).eps),
    )

    # Replacer chaque candidat sur le point le plus marqué du signal ECG filtré.
    # Chercher le point ECG le plus marqué dans une fenêtre de ±0,08 seconde.
    radius = round(0.08 * fs)
    qrs_peaks = []
    for candidate in candidates:
        left = max(0, candidate - radius)
        right = min(len(filtered), candidate + radius + 1)
        qrs_peaks.append(left + int(np.argmax(np.abs(filtered[left:right]))))

    # Renvoyer les positions QRS affinées, les pics d’énergie et les signaux à tracer.
    return np.asarray(qrs_peaks, dtype=int), candidates, filtered, energy_smooth

# Adaptateur au format utilisé par le fichier de démarrage du jalon 4.
def detecter_qrs(x, fs=FS):
    pics, candidats, _, _ = detect_qrs(x, fs=fs)
    return pics, candidats

N_BATT = 40
def suite_battements(x):
    # Une ligne = intervalle RR (secondes) et amplitude QRS (mV).
    pics, _ = detecter_qrs(x)
    rr = np.diff(pics).astype('float32') / FS
    amplitude_qrs = x[pics[1:]].astype('float32')
    sequence = np.zeros((N_BATT, 2), dtype='float32')
    n = min(len(rr), N_BATT)
    if n:
        sequence[:n] = np.column_stack([rr, amplitude_qrs])[:n]
    return sequence

BATT = {
    split: np.stack([suite_battements(x) for x in A[f'X_{split}']])
    for split in ['train', 'val']
}
print('Forme train :', BATT['train'].shape, '(fenêtres, 40 intervalles max, 2 caractéristiques)')
print('Forme validation :', BATT['val'].shape)


Forme train : (8400, 40, 2) (fenêtres, 40 intervalles max, 2 caractéristiques)
Forme validation : (1800, 40, 2)


## 18. Garder les zéros comme marqueurs de remplissage

Les zéros ajoutés après le dernier battement doivent rester des zéros pour que `Masking(mask_value=0)` les reconnaisse. Nous standardisons RR et amplitude en utilisant seulement les lignes réelles du train, puis réappliquons ces mêmes statistiques à la validation. Les lignes de remplissage restent à zéro.


In [31]:
# Une position est réelle si RR ou amplitude n’est pas le zéro de remplissage.
MASQUES_BATT = {s: np.any(BATT[s] != 0.0, axis=-1) for s in ['train', 'val']}
positions_train = MASQUES_BATT['train']
assert positions_train.any(), 'Aucun battement détecté dans le train.'

MOY_BATT = BATT['train'][positions_train].mean(axis=0)
ECART_BATT = BATT['train'][positions_train].std(axis=0) + 1e-6
BATT_NORM = {}
for split in ['train', 'val']:
    # Préinitialiser à zéro conserve le marqueur de remplissage pour le masquage.
    BATT_NORM[split] = np.zeros_like(BATT[split], dtype='float32')
    masque = MASQUES_BATT[split]
    BATT_NORM[split][masque] = (BATT[split][masque] - MOY_BATT) / ECART_BATT

nb_positions = MASQUES_BATT['train'].sum(axis=1)
print('Nombre de lignes réelles par fenêtre (train) — min/médiane/max :',
      int(nb_positions.min()), float(np.median(nb_positions)), int(nb_positions.max()))
print('Valeurs moyennes / écarts-types train (RR, amplitude) :',
      np.round(MOY_BATT, 4), '/', np.round(ECART_BATT, 4))
print('Contrôle : toutes les positions de padding sont nulles :',
      bool(np.all(BATT_NORM['train'][~MASQUES_BATT['train']] == 0)))


Nombre de lignes réelles par fenêtre (train) — min/médiane/max : 5 12.0 22
Valeurs moyennes / écarts-types train (RR, amplitude) : [0.71   1.0696] / [0.2327 0.4764]
Contrôle : toutes les positions de padding sont nulles : True


## 19. Construire le GRU avec masquage

`Masking` indique au GRU d’ignorer les lignes `(0, 0)` ajoutées pour compléter les fenêtres plus courtes. Le GRU lit les lignes valides dans l’ordre, résume la suite, puis la couche dense prédit les cinq classes.


In [33]:
GRAINE_GRU = 46
keras.utils.set_random_seed(GRAINE_GRU)

entree_batt = keras.Input(shape=(N_BATT, 2), name='suite_RR_QRS')
x_batt = layers.Masking(mask_value=0.0, name='ignorer_padding')(entree_batt)
x_batt = layers.GRU(32, name='gru_battements')(x_batt)
sortie_batt = layers.Dense(5, activation='softmax', name='rythme')(x_batt)
modele_gru_batt = keras.Model(entree_batt, sortie_batt, name='gru_suite_battements')
modele_gru_batt.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                        loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print('Entrée GRU :', modele_gru_batt.input_shape)
print('Paramètres :', modele_gru_batt.count_params())
modele_gru_batt.summary()


Entrée GRU : (None, 40, 2)
Paramètres : 3621
Model: "gru_suite_battements"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ suite_RR_QRS        │ (None, 40, 2)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 40, 2)     │          0 │ suite_RR_QRS[0][… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ignorer_padding     │ (None, 40, 2)     │          0 │ suite_RR_QRS[0][… │
│ (Masking)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│

## 20. Entraîner et comparer les familles

Nous utilisons les mêmes fenêtres train/validation, les poids de classes et trois époques exploratoires. Le F1 macro et les rappels par classe sont calculés sur la validation. Nous enregistrons aussi les paramètres et le temps d’entraînement par époque. Toujours aucun accès au test.


In [35]:
class F1MacroValidationBattements(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(BATT_NORM['val'], batch_size=256, verbose=0)
        logs['val_f1_macro'] = f1_score(A['classe_val'], p.argmax(axis=1),
                                         average='macro', zero_division=0)

class TempsEntrainementEpoque(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self.debut = perf_counter()
        self.mesure_faite = False
    def on_test_begin(self, logs=None):
        if not self.mesure_faite:
            self.secondes.append(perf_counter() - self.debut)
            self.mesure_faite = True
    def on_train_begin(self, logs=None):
        self.secondes = []
    def on_epoch_end(self, epoch, logs=None):
        if not self.mesure_faite:
            self.secondes.append(perf_counter() - self.debut)

temps_gru = TempsEntrainementEpoque()
callback_f1_batt = F1MacroValidationBattements()
historique_gru_batt = modele_gru_batt.fit(
    BATT_NORM['train'], A['classe_train'].astype('int32'),
    validation_data=(BATT_NORM['val'], A['classe_val'].astype('int32')),
    class_weight=POIDS_CLASSES, epochs=3, batch_size=128, verbose=0,
    callbacks=[callback_f1_batt, temps_gru],
)
proba_gru_batt = modele_gru_batt.predict(BATT_NORM['val'], batch_size=256, verbose=0)
pred_gru_batt = proba_gru_batt.argmax(axis=1)
F1_GRU_BATT = f1_score(A['classe_val'], pred_gru_batt, average='macro', zero_division=0)
ACC_GRU_BATT = accuracy_score(A['classe_val'], pred_gru_batt)
RAPPELS_GRU_BATT = recall_score(A['classe_val'], pred_gru_batt, labels=CLASSES,
                               average=None, zero_division=0)
TEMPS_GRU_BATT = float(np.mean(temps_gru.secondes))
print('GRU suite de battements — validation Holter A')
print('F1 macro :', round(F1_GRU_BATT, 3), '; exactitude :', round(ACC_GRU_BATT, 3))
print('Rappels classes 0 à 4 :', np.round(RAPPELS_GRU_BATT, 3))
print('Paramètres :', modele_gru_batt.count_params(),
      '; secondes/époque :', round(TEMPS_GRU_BATT, 2))

# Consigner cette expérience, y compris la limite liée au détecteur reconstruit.
import csv
with JOURNAL.open(encoding='utf-8-sig', newline='') as f:
    lignes_journal = list(csv.DictReader(f))
champs_journal = ['jalon','experience','graine','f1_macro_validation','exactitude_validation',
                  'rappel_classes_0_a_4','parametres','secondes_par_epoque',
                  'observation_gradient','statut','notes']
nom_experience = 'GRU_suite_battements_40'
lignes_journal = [r for r in lignes_journal if r['experience'] != nom_experience]
lignes_journal.append({
    'jalon':4, 'experience':nom_experience, 'graine':GRAINE_GRU,
    'f1_macro_validation':round(float(F1_GRU_BATT),4),
    'exactitude_validation':round(float(ACC_GRU_BATT),4),
    'rappel_classes_0_a_4':' / '.join(f'{v:.3f}' for v in RAPPELS_GRU_BATT),
    'parametres':modele_gru_batt.count_params(),
    'secondes_par_epoque':round(TEMPS_GRU_BATT,2),
    'observation_gradient':'GRU avec masking ; diagnostic des gradients détaillé réalisé pour RNN et LSTM bruts',
    'statut':'exploratoire, 3 époques',
    'notes':'RR + amplitude QRS ; détecteur QRS reconstruit car uc3_demarrage.ipynb absent ; test fermé'})
with JOURNAL.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=champs_journal)
    writer.writeheader(); writer.writerows(lignes_journal)


GRU suite de battements — validation Holter A
F1 macro : 0.581 ; exactitude : 0.6
Rappels classes 0 à 4 : [0.552 0.286 0.671 0.788 0.933]
Paramètres : 3621 ; secondes/époque : 1.24


## 21. Lecture provisoire des trois familles

La famille **RNN/LSTM sur signal brut** reçoit 1 250 valeurs par fenêtre. La famille **CNN** apprend des motifs locaux ; l’hybride ajoute l’ordre temporel après réduction. La famille **GRU sur battements** reçoit au plus 40 paires RR–amplitude, avec les lignes de remplissage masquées.

Comparez les rappels par classe, le F1 macro, les paramètres et le temps par époque. Cette expérience GRU dépend du détecteur QRS reconstruit, limite à mentionner dans la page de décisions. La décision et le test final restent à faire.


## 22. Vérifications théoriques demandées

**Disparition du gradient.** Si le gradient est multiplié par le même facteur à chacun des 1 250 pas, il est multiplié au total par `facteur ** 1250`. Un résultat très proche de zéro signifie que les premiers instants transmettent très peu de signal d’apprentissage jusqu’à la prédiction.

**Paramètres du LSTM.** Avec une variable d’entrée et 32 unités, Keras apprend quatre transformations (portes et état candidat). Chaque transformation possède des poids d’entrée, des poids récurrents et un biais. La formule Keras est `4 × unités × (entrée + unités + 1)`.


In [38]:
# Vérifier les deux facteurs répétés sur les 1 250 pas temporels.
reste_facteur_09 = 0.9 ** LONGUEUR_FENETRE
reste_facteur_099 = 0.99 ** LONGUEUR_FENETRE
print('0,9 puissance 1 250 =', f'{reste_facteur_09:.3e}')
print('0,99 puissance 1 250 =', f'{reste_facteur_099:.3e}')

# Formule théorique du LSTM Keras : 4 portes × unités × (entrée + unités + biais).
unites_theorie = 32
variables_entree = 1
parametres_formule = 4 * unites_theorie * (variables_entree + unites_theorie + 1)

# Vérifier le résultat avec une vraie couche Keras, sans couche de classification.
modele_lstm_theorie = keras.Sequential([
    keras.Input(shape=(None, variables_entree)),
    layers.LSTM(unites_theorie),
])
parametres_keras = modele_lstm_theorie.count_params()
print('Paramètres par formule :', parametres_formule)
print('Paramètres comptés par Keras :', parametres_keras)
assert parametres_formule == parametres_keras == 4352


0,9 puissance 1 250 = 6.355e-58
0,99 puissance 1 250 = 3.499e-06
Paramètres par formule : 4352
Paramètres comptés par Keras : 4352


## 23. Interpréter les vérifications et le masquage

Pour `0,9`, il ne reste qu’environ `6,36 × 10⁻⁵⁸` du gradient initial après 1 250 multiplications : l’information d’apprentissage des premiers instants devient pratiquement nulle dans cette hypothèse. Pour `0,99`, il reste environ `3,5 × 10⁻⁶` : c’est plus grand, mais encore très faible. C’est pourquoi une longue séquence brute est difficile pour un RNN simple. Un LSTM ou GRU utilise des portes pour mieux contrôler la circulation de l’information, sans garantir à lui seul un bon résultat.

Le LSTM théorique compte **4 352 paramètres**. Ce chiffre concerne une entrée à une variable et 32 unités, sans couche de classification.

Dans la suite de battements, le padding ajoute `(0, 0)` pour obtenir une longueur fixe de 40. `Masking(mask_value=0)` indique au GRU d’ignorer ces lignes : sans cela, il pourrait les interpréter comme de faux battements et leur ordre modifierait le résumé de la séquence.


## 24. Comparaison de validation plus fiable

Les essais précédents duraient seulement trois époques et servaient à explorer. Pour éviter de choisir un modèle parce qu’il apprend plus vite au tout début, nous reprenons les trois candidats principaux — CNN seul, CNN + LSTM et GRU sur battements — avec une initialisation neuve et la même limite de 12 époques.

L’arrêt anticipé suit le F1 macro de validation et restaure les meilleurs poids. Les données restent identiques ; l’ensemble test reste fermé. Le RNN simple et le LSTM brut restent des références exploratoires du premier volet.


In [41]:
class ArretMeilleurF1(keras.callbacks.Callback):
    # Mémoriser le meilleur modèle observé sur la validation et arrêter après 3 époques sans progrès.
    def __init__(self, patience=3):
        super().__init__()
        self.patience = patience
        self.meilleur = -np.inf
        self.attente = 0
        self.meilleurs_poids = None
        self.meilleure_epoque = 0
    def on_epoch_end(self, epoch, logs=None):
        score = logs.get('val_f1_macro', -np.inf)
        if score > self.meilleur:
            self.meilleur = score
            self.attente = 0
            self.meilleure_epoque = epoch + 1
            self.meilleurs_poids = self.model.get_weights()
        else:
            self.attente += 1
            if self.attente >= self.patience:
                self.model.stop_training = True
    def on_train_end(self, logs=None):
        if self.meilleurs_poids is not None:
            self.model.set_weights(self.meilleurs_poids)

def creer_cnn_validation():
    entree = keras.Input(shape=(LONGUEUR_FENETRE, 1), name='ecg_cnn_final')
    seq = construire_extracteur_cnn(entree, 'cnn_validation')
    sortie = layers.Dense(5, activation='softmax')(layers.GlobalAveragePooling1D()(seq))
    modele = keras.Model(entree, sortie, name='cnn_seul_validation')
    modele.compile(optimizer=keras.optimizers.Adam(1e-3),
                   loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return modele

def creer_hybride_validation():
    entree = keras.Input(shape=(LONGUEUR_FENETRE, 1), name='ecg_hybride_final')
    seq = construire_extracteur_cnn(entree, 'cnn_lstm_validation')
    sortie = layers.Dense(5, activation='softmax')(layers.LSTM(16)(seq))
    modele = keras.Model(entree, sortie, name='cnn_lstm_validation')
    modele.compile(optimizer=keras.optimizers.Adam(1e-3),
                   loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return modele

def creer_gru_validation():
    entree = keras.Input(shape=(N_BATT, 2), name='suite_battements_finale')
    seq = layers.Masking(mask_value=0.0)(entree)
    sortie = layers.Dense(5, activation='softmax')(layers.GRU(32)(seq))
    modele = keras.Model(entree, sortie, name='gru_battements_validation')
    modele.compile(optimizer=keras.optimizers.Adam(1e-3),
                   loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return modele

def comparer_validation_approfondie(nom, constructeur, Xtr, Xva, ytr, yva, graine,
                                    sequence_battements=False):
    keras.utils.set_random_seed(graine)
    modele = constructeur()
    f1_callback = (F1MacroValidationBattements() if sequence_battements
                   else F1MacroValidation())
    temps_callback = TempsEntrainementEpoque()
    arret = ArretMeilleurF1(patience=3)
    historique = modele.fit(
        Xtr, ytr.astype('int32'), validation_data=(Xva, yva.astype('int32')),
        class_weight=POIDS_CLASSES, epochs=12, batch_size=128, verbose=0,
        callbacks=[f1_callback, temps_callback, arret],
    )
    proba = modele.predict(Xva, batch_size=256, verbose=0)
    predictions = proba.argmax(axis=1)
    rappels = recall_score(yva, predictions, labels=CLASSES, average=None, zero_division=0)
    exactitude = accuracy_score(yva, predictions)
    print(f'\n{nom} : meilleur F1 macro={arret.meilleur:.3f} à époque {arret.meilleure_epoque};',
          f'{len(historique.history["loss"])} époques effectuées')
    print('Exactitude :', round(exactitude,3), '; rappels classes 0–4 :', np.round(rappels,3))
    print('Paramètres :', modele.count_params(),
          '; secondes/époque :', round(float(np.mean(temps_callback.secondes)),2))
    ligne = {
        'jalon':4, 'experience':nom, 'graine':graine,
        'f1_macro_validation':round(float(arret.meilleur),4),
        'exactitude_validation':round(float(exactitude),4),
        'rappel_classes_0_a_4':' / '.join(f'{v:.3f}' for v in rappels),
        'parametres':modele.count_params(),
        'secondes_par_epoque':round(float(np.mean(temps_callback.secondes)),2),
        'observation_gradient':'F1 macro de validation surveillé ; meilleur poids restauré',
        'statut':f'validation, plafond 12 époques atteint ({len(historique.history["loss"])} époques)',
        'notes':'train/validation Holter A ; nouvelle graine ; test fermé'
    }
    return ligne

RESULTATS_VALIDATION_APPROFONDIE = [
    comparer_validation_approfondie('CNN_seul_validation_affinee', creer_cnn_validation,
        SA['train'], SA['val'], A['classe_train'], A['classe_val'], 47),
    comparer_validation_approfondie('CNN_suivi_LSTM_validation_affinee', creer_hybride_validation,
        SA['train'], SA['val'], A['classe_train'], A['classe_val'], 48),
    comparer_validation_approfondie('GRU_battements_validation_affinee', creer_gru_validation,
        BATT_NORM['train'], BATT_NORM['val'], A['classe_train'], A['classe_val'], 49,
        sequence_battements=True),
]

# Conserver les essais exploratoires et ajouter/remplacer les lignes de cette étape contrôlée.
import csv
with JOURNAL.open(encoding='utf-8-sig', newline='') as f:
    lignes_journal = list(csv.DictReader(f))
champs_journal = ['jalon','experience','graine','f1_macro_validation','exactitude_validation',
                  'rappel_classes_0_a_4','parametres','secondes_par_epoque',
                  'observation_gradient','statut','notes']
noms_affines = {r['experience'] for r in RESULTATS_VALIDATION_APPROFONDIE}
lignes_journal = [r for r in lignes_journal if r['experience'] not in noms_affines]
lignes_journal.extend(RESULTATS_VALIDATION_APPROFONDIE)
with JOURNAL.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=champs_journal)
    writer.writeheader(); writer.writerows(lignes_journal)



CNN_seul_validation_affinee : meilleur F1 macro=0.595 à époque 12; 12 époques effectuées
Exactitude : 0.576 ; rappels classes 0–4 : [0.434 0.697 0.866 0.682 0.533]
Paramètres : 10885 ; secondes/époque : 10.44

CNN_suivi_LSTM_validation_affinee : meilleur F1 macro=0.420 à époque 11; 12 époques effectuées
Exactitude : 0.422 ; rappels classes 0–4 : [0.234 0.214 0.719 0.949 0.567]
Paramètres : 15829 ; secondes/époque : 14.59

GRU_battements_validation_affinee : meilleur F1 macro=0.822 à époque 12; 12 époques effectuées
Exactitude : 0.821 ; rappels classes 0–4 : [0.762 0.761 0.918 0.899 0.95 ]
Paramètres : 3621 ; secondes/époque : 0.97


## 25. Décider à partir de la validation

Retenir un modèle exige plus qu’un seul F1 : lire les rappels de toutes les classes, vérifier le coût en paramètres et le temps, puis comparer aux objectifs du système. Le meilleur F1 macro de cette étape guidera le choix, mais les limites du détecteur QRS reconstruit et les performances de chaque classe restent déterminantes. Le test ne doit être consulté qu’une fois le choix verrouillé.


## 26. Vérifier les cibles système sur validation

Le sujet cite trois objectifs finaux : F1 macro ≥ 0,97, sensibilité FA à 95 % de spécificité, et modèle ≤ 2 Mo avec latence ≤ 100 ms par fenêtre. Ils ne sont pas exigés à la fin du jalon, mais il faut mesurer l’écart. Nous calculons le seuil FA sur les scores non-FA de **validation seulement**. Nous séparons aussi la latence du GRU seul et le temps total avec détection des QRS, car cette représentation dépend d’un prétraitement.

Nous réentraînons le GRU candidat avec la graine et le protocole de l’affinement (12 époques, meilleurs poids selon F1 macro) afin d’enregistrer exactement le modèle et ses mesures. Le test reste fermé.


In [44]:
# Reproduire le GRU candidat (graine 49) et conserver son meilleur modèle de validation.
GRAINE_MODELE_RETENU = 49
keras.utils.set_random_seed(GRAINE_MODELE_RETENU)
modele_gru_retenu = creer_gru_validation()
f1_batt_retenu = F1MacroValidationBattements()
temps_batt_retenu = TempsEntrainementEpoque()
arret_batt_retenu = ArretMeilleurF1(patience=3)
historique_gru_retenu = modele_gru_retenu.fit(
    BATT_NORM['train'], A['classe_train'].astype('int32'),
    validation_data=(BATT_NORM['val'], A['classe_val'].astype('int32')),
    class_weight=POIDS_CLASSES, epochs=12, batch_size=128, verbose=0,
    callbacks=[f1_batt_retenu, temps_batt_retenu, arret_batt_retenu],
)

# Calculer les résultats multi-classes à l’argmax sur validation.
PROBA_VAL_GRU_RETENU = modele_gru_retenu.predict(BATT_NORM['val'], batch_size=256, verbose=0)
PRED_VAL_GRU_RETENU = PROBA_VAL_GRU_RETENU.argmax(axis=1)
F1_VAL_GRU_RETENU = f1_score(A['classe_val'], PRED_VAL_GRU_RETENU,
                             average='macro', zero_division=0)
ACC_VAL_GRU_RETENU = accuracy_score(A['classe_val'], PRED_VAL_GRU_RETENU)
RAPPELS_VAL_GRU_RETENU = recall_score(A['classe_val'], PRED_VAL_GRU_RETENU, labels=CLASSES,
                                      average=None, zero_division=0)

# Choisir le seuil le plus bas qui garde au moins 95 % de spécificité sur les non-FA.
SCORES_NON_FA = PROBA_VAL_GRU_RETENU[A['classe_val'] != 1, 1]
SCORES_FA = PROBA_VAL_GRU_RETENU[A['classe_val'] == 1, 1]
seuils_candidats = np.unique(SCORES_NON_FA)
spec_par_seuil = np.array([np.mean(SCORES_NON_FA < seuil) for seuil in seuils_candidats])
seuils_valides = seuils_candidats[spec_par_seuil >= 0.95]
SEUIL_FA_VAL_GRU = float(seuils_valides[0]) if len(seuils_valides) else float(np.nextafter(SCORES_NON_FA.max(), np.inf))
SPEC_FA_VAL_GRU = float(np.mean(SCORES_NON_FA < SEUIL_FA_VAL_GRU))
SENS_FA_VAL_GRU = float(np.mean(SCORES_FA >= SEUIL_FA_VAL_GRU))

# Sauvegarder le modèle candidat afin de conserver précisément les poids évalués.
DOSSIER_MODELES = DELIVERABLE_DIR / 'modeles'
DOSSIER_MODELES.mkdir(parents=True, exist_ok=True)
FICHIER_MODELE_GRU = DOSSIER_MODELES / 'gru_battements_jalon4_retenu.keras'
modele_gru_retenu.save(FICHIER_MODELE_GRU)
TAILLE_MODELE_OCTETS = FICHIER_MODELE_GRU.stat().st_size
TAILLE_POIDS_MIO = modele_gru_retenu.count_params() * 4 / (1024 ** 2)

# Mesurer la latence du réseau seul après échauffement, sur une fenêtre à la fois.
@tf.function
def inferer_gru_un_exemple(x):
    return modele_gru_retenu(x, training=False)
exemple_val = tf.convert_to_tensor(BATT_NORM['val'][:1])
for _ in range(20):
    inferer_gru_un_exemple(exemple_val).numpy()
N_REPETITIONS = 500
debut = perf_counter()
for _ in range(N_REPETITIONS):
    inferer_gru_un_exemple(exemple_val).numpy()
LATENCE_GRU_MS = (perf_counter() - debut) * 1000 / N_REPETITIONS

# Mesurer aussi la chaîne QRS→RR/amplitude→normalisation→GRU sur 100 fenêtres validation.
def preparer_une_suite(x_brut):
    seq = suite_battements(x_brut)[None, ...]
    masque = np.any(seq != 0.0, axis=-1)
    seq_norm = np.zeros_like(seq, dtype='float32')
    seq_norm[masque] = (seq[masque] - MOY_BATT) / ECART_BATT
    return tf.convert_to_tensor(seq_norm)
N_LATENCE_BOUT_EN_BOUT = min(100, len(A['X_val']))
debut = perf_counter()
for x_brut in A['X_val'][:N_LATENCE_BOUT_EN_BOUT]:
    inferer_gru_un_exemple(preparer_une_suite(x_brut)).numpy()
LATENCE_TOTALE_MS = (perf_counter() - debut) * 1000 / N_LATENCE_BOUT_EN_BOUT

print('GRU candidat — validation Holter A')
print('F1 macro / exactitude :', round(F1_VAL_GRU_RETENU,3), '/', round(ACC_VAL_GRU_RETENU,3))
print('Rappels classes 0 à 4 :', np.round(RAPPELS_VAL_GRU_RETENU,3))
print('Seuil FA choisi sur non-FA validation :', round(SEUIL_FA_VAL_GRU,5))
print('Spécificité FA validation :', round(SPEC_FA_VAL_GRU,3),
      '; sensibilité FA validation :', round(SENS_FA_VAL_GRU,3))
print('Paramètres :', modele_gru_retenu.count_params(),
      '; poids float32 théoriques :', round(TAILLE_POIDS_MIO,4), 'Mio')
print('Fichier modèle enregistré :', FICHIER_MODELE_GRU)
print('Taille fichier modèle :', TAILLE_MODELE_OCTETS, 'octets')
print('Latence GRU seul, batch 1 :', round(LATENCE_GRU_MS,3), 'ms/fenêtre')
print('QRS + préparation + GRU, batch 1 :', round(LATENCE_TOTALE_MS,3), 'ms/fenêtre (100 fenêtres)')


GRU candidat — validation Holter A
F1 macro / exactitude : 0.822 / 0.821
Rappels classes 0 à 4 : [0.762 0.761 0.918 0.899 0.95 ]
Seuil FA choisi sur non-FA validation : 0.64198
Spécificité FA validation : 0.95 ; sensibilité FA validation : 0.592
Paramètres : 3621 ; poids float32 théoriques : 0.0138 Mio
Fichier modèle enregistré : /home/mapster/ABOUBAKER/SDIA_PRACTICES/SCHOOL/s3-s4/advanced-ai/USECASE_TP/PRACTICE/deliverables/Jalon4/modeles/gru_battements_jalon4_retenu.keras
Taille fichier modèle : 73104 octets
Latence GRU seul, batch 1 : 1.114 ms/fenêtre
QRS + préparation + GRU, batch 1 : 2.699 ms/fenêtre (100 fenêtres)


## 27. Interpréter les objectifs sans ouvrir le test

Le F1 macro, le seuil d’alerte FA, la taille et la latence sont mesurés ici sur validation. Ces résultats servent à comparer le candidat aux objectifs du système ; ils ne sont pas une évaluation finale. La latence du réseau seul exclut la détection QRS ; la mesure bout en bout l’inclut avec le détecteur reconstruit. Le résultat le plus important avant le choix est de vérifier si le seuil atteint bien 95 % de spécificité et quelle sensibilité il donne.

Une fois le choix et le seuil verrouillés, le test caché ne sera utilisé qu’une fois.


## 28. Choix verrouillé avant l’évaluation finale

Pour le test, nous retenons le GRU sur suite de battements : c’est le meilleur candidat Jalon 4 sur la validation affinée (F1 macro 0,822 ; rappel de chaque classe ≥ 0,761) avec 3 621 paramètres. La sensibilité FA à 95 % de spécificité reste insuffisante (0,592), mais le sujet demande de mesurer l’écart et de le justifier ; il ne faut pas régler le seuil sur le test. Le seuil fixé sur validation est **0,64198**.

**Transparence :** le même jeu test Holter A a déjà été rapporté dans le travail du Jalon 1. Cette cellule effectue l’évaluation finale du modèle Jalon 4 une fois, mais le partage n’est donc plus inédit à l’échelle de tout le projet. Ne modifier ni modèle ni seuil en fonction de ces résultats.


## 29. Évaluer une seule fois sur le test

Cette cellule charge maintenant les fenêtres de test, transforme chaque fenêtre avec le détecteur QRS et les statistiques apprises sur le train, puis charge le modèle et le seuil verrouillés. Elle calcule les métriques multi-classes et l’alerte FA. N’exécutez pas cette cellule une seconde fois pour comparer ou modifier des modèles.


In [48]:
# Garde-fou contre une seconde exécution dans la même session du notebook.
if globals().get('TEST_JALON4_DEJA_EVALUE', False):
    raise RuntimeError('Le test Jalon 4 a déjà été évalué dans cette session ; ne pas le relancer.')
TEST_JALON4_DEJA_EVALUE = True

cles_test = ['X_test', 'classe_test', 'patient_test']
with np.load(PRACTICE_DIR / 'data' / 'holter_A' / 'uc3_holterA.npz') as archive:
    DONNEES_TEST_J4 = {cle: archive[cle].copy() for cle in cles_test}
X_TEST_BRUT = DONNEES_TEST_J4['X_test']
y_test_j4 = DONNEES_TEST_J4['classe_test'].astype('int32')
patients_test_j4 = DONNEES_TEST_J4['patient_test']

# Vérifier que les patients test ne recouvrent pas train/validation.
assert not (set(np.unique(patients_test_j4)) &
            (set(np.unique(A['patient_train'])) | set(np.unique(A['patient_val']))))
BATT_TEST = np.stack([suite_battements(x) for x in X_TEST_BRUT])
MASQUE_TEST = np.any(BATT_TEST != 0.0, axis=-1)
BATT_TEST_NORM = np.zeros_like(BATT_TEST, dtype='float32')
BATT_TEST_NORM[MASQUE_TEST] = (BATT_TEST[MASQUE_TEST] - MOY_BATT) / ECART_BATT

modele_gru_test = keras.models.load_model(FICHIER_MODELE_GRU)
proba_test_j4 = modele_gru_test.predict(BATT_TEST_NORM, batch_size=256, verbose=0)
pred_test_j4 = proba_test_j4.argmax(axis=1)
F1_MACRO_TEST_J4 = f1_score(y_test_j4, pred_test_j4, average='macro', zero_division=0)
EXACTITUDE_TEST_J4 = accuracy_score(y_test_j4, pred_test_j4)
RAPPELS_TEST_J4 = recall_score(y_test_j4, pred_test_j4, labels=CLASSES,
                              average=None, zero_division=0)

# Appliquer le seuil FA choisi exclusivement sur validation.
SEUIL_FA_VERROUILLE = 0.64198
alerte_fa_test = proba_test_j4[:, 1] >= SEUIL_FA_VERROUILLE
fa_reelle_test = y_test_j4 == 1
SENS_FA_TEST_J4 = float(np.mean(alerte_fa_test[fa_reelle_test]))
SPEC_FA_TEST_J4 = float(np.mean(~alerte_fa_test[~fa_reelle_test]))
print('ÉVALUATION FINALE UNIQUE — TEST HOLTER A')
print('Fenêtres / patients :', len(y_test_j4), '/', len(np.unique(patients_test_j4)))
print('F1 macro :', round(F1_MACRO_TEST_J4,3), '; exactitude :', round(EXACTITUDE_TEST_J4,3))
print('Rappels classes 0 à 4 :', np.round(RAPPELS_TEST_J4,3))
print('Seuil FA fixé sur validation :', SEUIL_FA_VERROUILLE)
print('Sensibilité FA test :', round(SENS_FA_TEST_J4,3),
      '; spécificité FA test :', round(SPEC_FA_TEST_J4,3))
print('Aucun choix ni ajustement ne doit être fait à partir de ces résultats.')

# Ajouter une seule ligne de résultat final au journal.
import csv
with JOURNAL.open(encoding='utf-8-sig', newline='') as f:
    lignes_journal = list(csv.DictReader(f))
champs_journal = ['jalon','experience','graine','f1_macro_validation','exactitude_validation',
                  'rappel_classes_0_a_4','parametres','secondes_par_epoque',
                  'observation_gradient','statut','notes']
nom_test='GRU_test_final_unique'
lignes_journal=[r for r in lignes_journal if r['experience'] != nom_test]
lignes_journal.append({
    'jalon':4,'experience':nom_test,'graine':49,
    'f1_macro_validation':'',
    'exactitude_validation':'',
    'rappel_classes_0_a_4':' / '.join(f'{v:.3f}' for v in RAPPELS_TEST_J4),
    'parametres':modele_gru_test.count_params(),'secondes_par_epoque':'',
    'observation_gradient':f'test final unique ; seuil FA validation={SEUIL_FA_VERROUILLE:.5f} ; sensibilité={SENS_FA_TEST_J4:.3f} ; spécificité={SPEC_FA_TEST_J4:.3f}',
    'statut':'test final calculé une fois ; ne pas réutiliser pour réglage',
    'notes':f'Test unique : F1 macro={F1_MACRO_TEST_J4:.4f} ; exactitude={EXACTITUDE_TEST_J4:.4f} ; rappels={" / ".join(f"{v:.3f}" for v in RAPPELS_TEST_J4)} ; même partition déjà rapportée au Jalon 1 ; détecteur QRS reconstruit'})
with JOURNAL.open('w',encoding='utf-8-sig',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=champs_journal)
    writer.writeheader(); writer.writerows(lignes_journal)


ÉVALUATION FINALE UNIQUE — TEST HOLTER A
Fenêtres / patients : 1800 / 60
F1 macro : 0.848 ; exactitude : 0.832
Rappels classes 0 à 4 : [0.776 0.721 0.98  0.934 0.949]
Seuil FA fixé sur validation : 0.64198
Sensibilité FA test : 0.554 ; spécificité FA test : 0.951
Aucun choix ni ajustement ne doit être fait à partir de ces résultats.


## 30. Après cette évaluation

Reporter les résultats test dans la page de décisions et expliquer tout écart avec les cibles. Cette partition ne doit plus servir à sélectionner un autre modèle ou seuil. La décision finale doit distinguer clairement les mesures validation, les mesures test et la limite liée à l’utilisation antérieure du même test dans le projet.
